# Chapitre 11 · Bien s'entraîner (solutions des exercices)

Ce notebook contient **uniquement les réponses aux quatre exercices** du notebook du chapitre.
Le code de la leçon, lui, vit dans le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

In [ ]:
# Mise en place (reprise de la leçon) : imports, corpus des fables et GPT du chapitre 10.
import math
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


def charger_corpus():
    """Lit le corpus des fables depuis le notebook du chapitre 10 ; repli inline sinon."""
    candidats = [
        Path("../partie_2_construire_le_cerveau/chapitre_10_le_transformer.ipynb"),
        Path("../../partie_2_construire_le_cerveau/chapitre_10_le_transformer.ipynb"),
    ]
    for chemin in candidats:
        if chemin.exists():
            nb = json.loads(chemin.read_text(encoding="utf-8"))
            for cellule in nb["cells"]:
                if cellule["cell_type"] == "code":
                    src = "".join(cellule["source"])
                    if src.strip().startswith("corpus"):
                        espace = {}
                        exec(src.split("chars =")[0], espace)
                        return espace["corpus"]
    # repli minimal (une fable) pour que le carnet tourne partout
    return ("LE CORBEAU ET LE RENARD\n"
            "Maitre corbeau, sur un arbre perche,\n"
            "Tenait en son bec un fromage.\n") * 200


corpus = charger_corpus()
chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in corpus])


block_size = 64


def fabriquer_batch(taille=32):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,))
    x = torch.stack([data[i : i + block_size] for i in ix])          # (B, T)
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])  # (B, T), decale d'un cran
    return x, y


def decouper_en_tetes(X, n_heads):
    B, T, d_model = X.shape
    d_k = d_model // n_heads
    return X.view(B, T, n_heads, d_k).transpose(1, 2)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, d_model = x.shape
        Q = decouper_en_tetes(self.W_Q(x), self.n_heads)
        K = decouper_en_tetes(self.W_K(x), self.n_heads)
        V = decouper_en_tetes(self.W_V(x), self.n_heads)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask[:T, :T] == 0, float("-inf"))
        poids = torch.softmax(scores, dim=-1)
        out = (poids @ V).transpose(1, 2).contiguous().view(B, T, d_model)
        return self.W_O(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x


d_model, n_heads, n_layers, d_ff = 96, 4, 2, 384


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.table_tokens = nn.Embedding(vocab_size, d_model)
        self.table_positions = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)
        self.register_buffer("masque", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T = x.shape
        h = self.table_tokens(x) + self.table_positions(torch.arange(T))
        for bloc in self.blocs:
            h = bloc(h, self.masque)
        return self.tete(self.ln_final(h))


STEPS = 800


def warmup_cosine(step, warmup_steps, total_steps, base_lr, min_lr=0.0):
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))


n_params = sum(p.numel() for p in GPT().parameters())
print(f"Mise en place OK : GPT {n_params} parametres | corpus {len(corpus)} caracteres")

### Exercice 1 · Gradient clipping — niveau ●

Ajoute le gradient clipping dans la boucle SGD ci-dessous. La ligne manquante,
`torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)`, va **entre** `loss.backward()` et
`opt.step()`. Elle renvoie la norme du gradient *avant* écrêtage (pratique pour l'observer).

In [ ]:
def entrainer_sgd_exo(lr, avec_clipping, steps=120):
    torch.manual_seed(42)
    modele = GPT()
    opt = torch.optim.SGD(modele.parameters(), lr=lr)
    journal = []
    for step in range(steps):
        x, y = fabriquer_batch()
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        opt.zero_grad()
        loss.backward()
        if avec_clipping:
            norme = torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)  # clippe et renvoie la norme
        else:
            norme = torch.nn.utils.clip_grad_norm_(modele.parameters(), 1e9)  # ne clippe rien
        opt.step()
        if step in (0, 10, 119):
            journal.append((step, loss.item(), norme.item()))
    return journal

In [ ]:
# Validation : gradient clipping.
sans = entrainer_sgd_exo(lr=2.0, avec_clipping=False)
avec = entrainer_sgd_exo(lr=2.0, avec_clipping=True)
print("sans clipping :", [(s, round(l, 1) if l == l else "nan") for s, l, _ in sans])
print("avec clipping :", [(s, round(l, 2)) for s, l, _ in avec])

# sans clipping, la loss finit en nan ; avec, elle reste finie et basse
assert sans[-1][1] != sans[-1][1], "sans clipping (lr=2.0), la loss doit exploser en nan"
assert avec[-1][1] < 3.5, "avec clipping, le run doit rester stable et descendre"
print("Exercice 1 OK : le clipping sauve un run qui, sans lui, diverge")

### Exercice 2 · Le schedule warmup + cosine — niveau ●●

Complète `warmup_cosine_exo`. Deux phases :
- si `step < warmup_steps` : montée **linéaire** de 0 à `base_lr` (donc `base_lr * step / warmup_steps`) ;
- sinon : descente en **cosinus** de `base_lr` vers `min_lr`, avec
  `progress = (step - warmup_steps) / (total_steps - warmup_steps)` et
  `min_lr + 0.5 * (base_lr - min_lr) * (1 + cos(pi * progress))`.

In [ ]:
def warmup_cosine_exo(step, warmup_steps, total_steps, base_lr, min_lr=0.0):
    if step < warmup_steps:
        # Phase 1 : montee lineaire de 0 a base_lr
        return base_lr * step / warmup_steps
    # Phase 2 : descente en cosinus de base_lr vers min_lr
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))

In [ ]:
# Validation : le schedule warmup + cosine.
assert abs(warmup_cosine_exo(0, 100, 800, 3e-2, 3e-3) - 0.0) < 1e-9, "au step 0, lr doit valoir 0"
assert abs(warmup_cosine_exo(50, 100, 800, 3e-2, 3e-3) - 1.5e-2) < 1e-9, "à mi-warmup, lr = base_lr/2"
assert abs(warmup_cosine_exo(100, 100, 800, 3e-2, 3e-3) - 3e-2) < 1e-9, "fin du warmup : lr = peak"
assert abs(warmup_cosine_exo(799, 100, 800, 3e-2, 3e-3) - 3e-3) < 1e-4, "fin : lr ~ min_lr"
print("Exercice 2 OK : schedule warmup + cosine correct")

### Exercice 3 · Accumulation de gradient — niveau ●●

Complète la boucle d'accumulation : pour chaque micro-batch, calcule la loss, **divise-la** par
`accumulation_steps` (pour moyenner, pas sommer), puis `backward()`. Ne remets **pas** les gradients à
zéro entre les micro-batchs : PyTorch les additionne, c'est le but.

In [ ]:
torch.manual_seed(0)
petit = nn.Linear(10, 3)
X = torch.randn(8, 10)
Y = torch.randint(0, 3, (8,))

# reference : un seul batch de 8
petit.zero_grad()
F.cross_entropy(petit(X), Y).backward()
grad_gros_batch = petit.weight.grad.clone()

# accumulation : 4 micro-batchs de 2
grand = nn.Linear(10, 3)
grand.load_state_dict(petit.state_dict())
grand.zero_grad()
accumulation_steps = 4
for i in range(accumulation_steps):
    xb, yb = X[i * 2:(i + 1) * 2], Y[i * 2:(i + 1) * 2]
    (F.cross_entropy(grand(xb), yb) / accumulation_steps).backward()  # on MOYENNE

In [ ]:
# Validation : accumulation de gradient.
grad_accumule = grand.weight.grad
assert grad_accumule is not None, "tu n'as pas encore fait de backward"
assert torch.allclose(grad_gros_batch, grad_accumule, atol=1e-6), \
    "l'accumulation doit donner le MÊME gradient qu'un gros batch (as-tu divisé par accumulation_steps ?)"
print("Exercice 3 OK : accumuler 4 micro-batchs == un batch 4x plus gros")

### Exercice 4 · La recette du labo, réunie — niveau ●●●

Complète la boucle d'entraînement : gradient clipping à 1.0 après le `backward`, puis `opt.step()`
et `sched.step()`. Ton run doit descendre nettement plus bas que le naïf (plateau ~2.4).

In [ ]:
def entrainer_labo_exo(lr=3e-2, warmup_steps=100):
    torch.manual_seed(42)
    modele = GPT()
    opt = torch.optim.AdamW(modele.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda s: warmup_cosine(s, warmup_steps, STEPS, 1.0, 0.1)
    )
    for step in range(STEPS):
        x, y = fabriquer_batch()
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
        opt.step()
        sched.step()
        if step % 200 == 0 or step == STEPS - 1:
            print(f"step {step:3d} | loss {loss.item():.2f} | lr {sched.get_last_lr()[0]:.4f}")
    return modele


gpt_labo_exo = entrainer_labo_exo()

In [ ]:
# Validation : la recette du labo doit finir bien en dessous du plateau naïf (~2.4).
g = torch.Generator().manual_seed(999)
gpt_labo_exo.eval()
with torch.no_grad():
    pertes = []
    for _ in range(20):
        ix = torch.randint(0, len(data) - block_size - 1, (32,), generator=g)
        x = torch.stack([data[i : i + block_size] for i in ix])
        y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
        pertes.append(F.cross_entropy(gpt_labo_exo(x).view(-1, vocab_size), y.view(-1)).item())
held = sum(pertes) / len(pertes)
print(f"loss held-out : {held:.2f}")
assert held < 1.8, "la recette du labo doit descendre bien sous le plateau naïf (~2.4)"
print("Exercice 4 OK : la recette du labo bat l'entraînement naïf, au même learning rate")